
# Семинар 3. Validation, regularization и categorical encoding
## Кейс: прогноз цены подержанных автомобилей

**Выстраиваем ML-процесс**: `preprocessing → validation → tuning → final evaluation`.

1. K-Fold cross-validation.
2. Сравним baseline, OLS, Ridge, Lasso и Elastic Net.
3. Сравним несколько способов кодирования категориальных переменных.




## Экономическая мотивация

Экономическая задача: по характеристикам автомобиля предсказать цену продажи. 


| Столбец | Описание |
|---|---|
| `listing_id` | Уникальный идентификатор объявления на Drom. |
| `brand` | Марка автомобиля: BMW, Chery, Lada и т. д. |
| `brand_group` | Группа марки: `Europe`, `China`, `Russia`. |
| `brand_country` | Страна происхождения марки, а не сборки автомобиля. |
| `model` | Модель автомобиля, например Tiguan. |
| `year` | Год выпуска. |
| `engine_volume_l` | Объём двигателя в литрах. |
| `power_hp` | Мощность двигателя в лошадиных силах. |
| `fuel_type` | Тип топлива или силовой установки: бензин, дизель, гибрид, электро и т. д. |
| `transmission` | Коробка передач: механика, АКПП, робот, вариатор. |
| `drive` | Привод: передний, задний, `4WD` — полный. |
| `mileage_km` | Пробег в километрах. |
| `body_type` | Тип кузова: седан, хэтчбек, внедорожник и т. д. |
| `trim` | Модификация и комплектация, указанные в объявлении. |
| `city` | Город объявления: Москва или Санкт-Петербург. |
| `price_rub` | **Цена предложения продавца в рублях**. Фактическая сумма сделки неизвестна. |
| `price_type` | Тип цены: `asking_price` — запрашиваемая продавцом цена. |
| `condition` | Категория автомобиля: `used` — с пробегом. |
| `is_damaged` | `1` — есть отметка о повреждении; `0` — такой отметки нет. Ноль не гарантирует отсутствие повреждений. |
| `labels` | Дополнительные отметки Drom, например «от собственника», «без пробега по РФ». |
| `attributes_raw` | Исходный текст характеристик из выдачи Drom. |
| `source_url` | Ссылка на конкретное объявление. |
| `search_url` | Страница выдачи, с которой скачано объявление. |
| `collected_at_utc` | Дата и время скачивания страницы в UTC, а не дата публикации объявления. |
| `model_rank` | Место модели внутри марки: от 1 до 5 по суммарному числу объявлений с пробегом в двух городах на момент отбора. |
| `model_id` | Идентификатор модели в каталоге Drom. |

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    train_test_split,
    KFold,
    RepeatedKFold,
    cross_validate,
    GridSearchCV,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error, make_scorer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

import statsmodels.api as sm


In [2]:
def calculate_metrics(y_true, y_pred, y_true_train, y_pred_train):
    metrics = {
        "train_rmse": root_mean_squared_error(y_true_train, y_pred_train),
        "test_rmse": root_mean_squared_error(y_true, y_pred),
        "train_mae": mean_absolute_error(y_true_train, y_pred_train),
        "test_mae": mean_absolute_error(y_true, y_pred),
        "train_mape": mean_absolute_percentage_error(y_true_train, y_pred_train),
        "test_mape": mean_absolute_percentage_error(y_true, y_pred),
        "train_r2": r2_score(y_true_train, y_pred_train),
        "test_r2": r2_score(y_true, y_pred)
    }

    return pd.DataFrame({'train': [metrics['train_rmse'], metrics['train_mae'], metrics['train_mape'], metrics['train_r2']],
                         'test': [metrics['test_rmse'], metrics['test_mae'], metrics['test_mape'], metrics['test_r2']]},
                        index=['RMSE', 'MAE', 'MAPE', 'R2']).round(3)


## 1. Загрузка данных и первичная настройка

In [7]:
DATA_PATH = 'data/drom_used_cars.csv'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head(2)


(8709, 26)


,listing_id,brand,brand_group,brand_country,model,year,engine_volume_l,power_hp,fuel_type,transmission,...,price_type,condition,is_damaged,labels,attributes_raw,source_url,search_url,collected_at_utc,model_rank,model_id
0,327415747,Volkswagen,Europe,Germany,Tiguan,2021,2.0,180.0,бензин,робот,...,asking_price,used,0,NaN,2.0 л (180 л.с.) | бензин | робот | 4WD | 70 0...,https://auto.drom.ru/moscow/volkswagen/tiguan/...,https://auto.drom.ru/moscow/volkswagen/tiguan/...,2026-09-18T08:36:30.751519+00:00,1,3711
1,312658259,Volkswagen,Europe,Germany,Tiguan,2021,2.0,180.0,бензин,робот,...,asking_price,used,0,NaN,2.0 л (180 л.с.) | бензин | робот | 4WD | 88 8...,https://auto.drom.ru/spb/volkswagen/tiguan/312...,https://auto.drom.ru/spb/volkswagen/tiguan/used/,2026-09-18T08:36:32.512797+00:00,1,3711


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8709 entries, 0 to 8708
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   listing_id        8709 non-null   int64  
 1   brand             8709 non-null   str    
 2   brand_group       8709 non-null   str    
 3   brand_country     8709 non-null   str    
 4   model             8709 non-null   str    
 5   year              8709 non-null   int64  
 6   engine_volume_l   8695 non-null   float64
 7   power_hp          8691 non-null   float64
 8   fuel_type         8686 non-null   str    
 9   transmission      8690 non-null   str    
 10  drive             8691 non-null   str    
 11  mileage_km        8709 non-null   int64  
 12  body_type         8595 non-null   str    
 13  trim              5273 non-null   str    
 14  city              8709 non-null   str    
 15  price_rub         8709 non-null   int64  
 16  price_type        8709 non-null   str    
 17  condit

# Preprocessing данных

In [ ]:
TARGET_COL = 'price_rub'
DROP_COLS = ['listing_id', 
'source_url', 
'search_url', 
'model_rank', 
'model_id', 
'collected_at_utc',
'price_type',
'condition',
'labels',
'trim',
'year']

# Создадим признак 'age' и удалим неиспользуемые признаки и пропущенные значения/
df['age'] = 2026-df['year']
df = df.drop(columns=DROP_COLS, errors='ignore').copy()
print('Размер датасета до удаления пропусков', df.shape)
df = df.dropna()
print('Размер датасета после удаления пропусков', df.shape)


feature_cols = [c for c in df.columns if c != TARGET_COL]
X = df[feature_cols].copy()
y = df[TARGET_COL].copy()/1000000


num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print('Числовые признаки:', num_cols)
print('Категориальные признаки:', cat_cols)



Размер датасета до удаления пропусков (8709, 16)
Размер датасета после удаления пропусков (8564, 16)
Числовые признаки: ['engine_volume_l', 'power_hp', 'mileage_km', 'is_damaged', 'age']
Категориальные признаки: ['brand', 'brand_group', 'brand_country', 'model', 'fuel_type', 'transmission', 'drive', 'body_type', 'city', 'attributes_raw']


In [10]:
X_copy = X[num_cols]
X_copy = sm.add_constant(X_copy)

model = sm.OLS(y, X_copy).fit(cov_type="HC3")

print(model.summary())



                            OLS Regression Results                            
Dep. Variable:              price_rub   R-squared:                       0.572
Model:                            OLS   Adj. R-squared:                  0.571
Method:                 Least Squares   F-statistic:                     385.0
Date:                Sun, 04 Oct 2026   Prob (F-statistic):               0.00
Time:                        13:40:57   Log-Likelihood:                -19536.
No. Observations:                8564   AIC:                         3.908e+04
Df Residuals:                    8558   BIC:                         3.913e+04
Df Model:                           5                                         
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -1.0865      0.157     

In [11]:
log_y = np.log(y)
model = sm.OLS(log_y, X_copy).fit(cov_type="HC3")
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              price_rub   R-squared:                       0.763
Model:                            OLS   Adj. R-squared:                  0.763
Method:                 Least Squares   F-statistic:                     3069.
Date:                Sun, 04 Oct 2026   Prob (F-statistic):               0.00
Time:                        13:41:00   Log-Likelihood:                -6189.6
No. Observations:                8564   AIC:                         1.239e+04
Df Residuals:                    8558   BIC:                         1.243e+04
Df Model:                           5                                         
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -0.0536      0.019     

## 2. Почему одного train/test split недостаточно

In [12]:
metrics = []
for seed in range(30):
    X_train, X_test, y_train, y_test = train_test_split(
        X[num_cols], y, test_size=0.2, random_state=seed
        )

    model = LinearRegression()

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_train = model.predict(X_train)
    seed_metrics = calculate_metrics(y_test, y_pred, y_train, y_pred_train)
    seed_metrics['seed'] = seed
    metrics.append(seed_metrics)

pd.concat(metrics).reset_index().groupby('index').agg({'test':['mean', 'std']})

test          
           mean       std
index                    
MAE    1.277800  0.038425
MAPE   1.098900  0.070369
R2     0.550600  0.028275
RMSE   2.436167  0.223259

Качество заметно меняется от `random_state`.   

Одна случайная test-выборка может дать слишком смещенную картину.


## 3. Baseline: посмотрим качество наивной модели предсказания среднего

In [13]:
help(DummyRegressor)

Help on class DummyRegressor in module sklearn.dummy:

class DummyRegressor(sklearn.base.MultiOutputMixin, sklearn.base.RegressorMixin, sklearn.base.BaseEstimator)
 |  DummyRegressor(*, strategy='mean', constant=None, quantile=None)
 |
 |  Regressor that makes predictions using simple rules.
 |
 |  This regressor is useful as a simple baseline to compare with other
 |  (real) regressors. Do not use it for real problems.
 |
 |  Read more in the :ref:`User Guide <dummy_estimators>`.
 |
 |  .. versionadded:: 0.13
 |
 |  Parameters
 |  ----------
 |  strategy : {"mean", "median", "quantile", "constant"}, default="mean"
 |      Strategy to use to generate predictions.
 |
 |      * "mean": always predicts the mean of the training set
 |      * "median": always predicts the median of the training set
 |      * "quantile": always predicts a specified quantile of the training set,
 |        provided with the quantile parameter.
 |      * "constant": always predicts a constant value that is prov

In [24]:
baseline = DummyRegressor(strategy='mean')
cv=KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'mape': 'neg_mean_absolute_percentage_error',
    'r2': 'r2',
}

result = cross_validate(baseline,
                        X[num_cols],
                        y,
                        cv=cv,
                        scoring=scoring)

pd.DataFrame(result)

,fit_time,score_time,test_rmse,test_mae,test_mape,test_r2
0,0.002726,0.004677,-3.363059,-1.853781,-2.360934,-0.000457
1,0.001369,0.002739,-3.259904,-1.775642,-2.173406,-0.001935
2,0.002152,0.002819,-3.971611,-1.912204,-2.036005,-0.000923
3,0.000912,0.001637,-3.694827,-1.920046,-2.560693,-0.000003
4,0.000937,0.001911,-3.759059,-1.909932,-2.301000,-0.000724


### сравним предсказания линейной регрессии с наивной

In [14]:
model = LinearRegression()

cv=KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'mape': 'neg_mean_absolute_percentage_error',
    'r2': 'r2',
}

result = cross_validate(model,
                        X[num_cols],
                        y,
                        cv=cv,
                        scoring=scoring)

pd.DataFrame(result)

,fit_time,score_time,test_rmse,test_mae,test_mape,test_r2
0,0.008722,0.002540,-2.175617,-1.235644,-1.143274,0.581308
1,0.004472,0.001938,-2.327598,-1.299293,-1.161464,0.489206
2,0.001600,0.001486,-2.625008,-1.309543,-0.994645,0.562751
3,0.002880,0.002426,-2.400092,-1.277301,-1.178871,0.578042
4,0.001568,0.001208,-2.450766,-1.256313,-1.021920,0.574637



## 5. Регуляризация: OLS, Ridge, Lasso, Elastic Net


- **OLS** минимизирует только ошибку подгонки;
- **Ridge** добавляет штраф на сумму квадратов коэффициентов;
- **Lasso** добавляет штраф на сумму модулей коэффициентов и может занулять часть из них;
- **Elastic Net** сочетает Ridge и Lasso.

Важно: перед регуляризацией числовые признаки обычно нужно **масштабировать**.


In [15]:
def cv_summary(estimator, X, y, cv, scoring):
    res_cols = ['test_rmse', 'test_mae', 'test_mape', 'test_r2']
    result = cross_validate(estimator, 
                            X, y, 
                            cv=cv, 
                            scoring=scoring, 
                            return_train_score=False, 
                            n_jobs=None)
    summary = pd.DataFrame(result)
    summary[['test_rmse', 'test_mae', 'test_mape']] = summary[['test_rmse', 'test_mae', 'test_mape']]*(-1)
    summary = summary[res_cols]

    return summary.rename(columns=lambda col: col.removeprefix("test_"))

In [16]:
# Compare OLS, Ridge, Lasso, ElasticNet 
cv = KFold(n_splits=5, shuffle=True, random_state=13)


models = {
    'OLS + scaler': Pipeline([
        ('preprocess', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Ridge + scaler': Pipeline([
        ('preprocess', StandardScaler()),
        ('model', Ridge(alpha=10.0))
    ]),
    'Lasso + scaler': Pipeline([
        ('preprocess', StandardScaler()),
        ('model', Lasso(alpha=0.01, max_iter=20000))
    ]),
    'ElasticNet + scaler': Pipeline([
        ('preprocess', StandardScaler()),
        ('model', ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000))
    ]),
}

rows = []
for name, est in models.items():
    summary = cv_summary(est, X[num_cols], y, cv=cv, scoring=scoring).mean()
    summary['model'] = name
    rows.append(summary)


model_compare = pd.DataFrame(rows).set_index('model').sort_values('rmse')
model_compare

,rmse,mae,mape,r2
model,,,,
Lasso + scaler,2.388493,1.270034,1.085345,0.562724
ElasticNet + scaler,2.389524,1.269465,1.085490,0.562375
Ridge + scaler,2.392834,1.274166,1.096915,0.561209
OLS + scaler,2.393248,1.275073,1.098888,0.561061


## 6. Подбор гиперпараметра с помощью GridSearchCV

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X[num_cols], y, test_size=0.2, random_state=42)

ridge_pipe = Pipeline([
    ('preprocess', StandardScaler()),
    ('model', Ridge())
])

ridge_grid = {
    'model__alpha': np.logspace(-3, 3, 13)
}

ridge_search = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_grid,
    scoring='neg_root_mean_squared_error',
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=None,
    refit=True,
)

ridge_search.fit(X_train, y_train)
print('Best alpha:', ridge_search.best_params_)
print('Best CV RMSE:', -ridge_search.best_score_)

best_ridge = ridge_search.best_estimator_
pred_test = best_ridge.predict(X_test)
print('Test RMSE:', root_mean_squared_error(y_test, pred_test))
print('Test MAE :', mean_absolute_error(y_test, pred_test))
print('Test R^2 :', r2_score(y_test, pred_test))

ridge_model = best_ridge.steps[-1][1]

# Названия признаков после предобработки
feature_names = best_ridge[:-1].get_feature_names_out()

coefficients = pd.Series(
    ridge_model.coef_,
    index=feature_names,
    name="coefficient",
).sort_values(key=abs, ascending=False)

display(coefficients)
print("Свободный член:", ridge_model.intercept_)


Best alpha: {'model__alpha': np.float64(316.22776601683796)}
Best CV RMSE: 2.5807422181359154
Test RMSE: 2.1689798213561384
Test MAE : 1.2069743355668083
Test R^2 : 0.5838588441930389


power_hp           2.360608
mileage_km        -0.688268
age               -0.221336
engine_volume_l    0.178129
is_damaged        -0.002901
Name: coefficient, dtype: float64

Свободный член: 2.671149996350898


## 7. Категориальные переменные 

В этом блоке сравним несколько encoding strategies:

1. **One-Hot Encoding** — раздувает размерность.
2. **Ordinal Encoding** — компактный, но создаёт искусственный порядок категорий.
3. **Frequency Encoding** — заменяет категорию её частотой в обучающей выборке.


In [18]:
ohe_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            num_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            cat_cols
        )
    ],
    verbose_feature_names_out=False
)

In [19]:
# Compare OLS, Ridge, Lasso, ElasticNet with One-Hot Encoding
cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'OLS + OHE': Pipeline([
        ('preprocess', ohe_preprocessor),
        ('model', LinearRegression())
    ]),
    'Ridge + OHE': Pipeline([
        ('preprocess', ohe_preprocessor),
        ('model', Ridge(alpha=10.0))
    ]),
    'Lasso + OHE': Pipeline([
        ('preprocess', ohe_preprocessor),
        ('model', Lasso(alpha=0.01, max_iter=20000))
    ]),
    'ElasticNet + OHE': Pipeline([
        ('preprocess', ohe_preprocessor),
        ('model', ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000))
    ]),
}

rows = []
for name, est in models.items():
    summary = cv_summary(est, X, y, cv=cv, scoring=scoring).mean()
    summary['model'] = name

    rows.append(summary)

model_compare = pd.DataFrame(rows).set_index('model').sort_values('rmse')
model_compare

,rmse,mae,mape,r2
model,,,,
Ridge + OHE,1.848243,0.891294,0.934432,0.735533
Lasso + OHE,1.930710,0.935591,0.950420,0.711816
ElasticNet + OHE,1.932732,0.940302,0.955828,0.711203
OLS + OHE,2.217246,1.106373,0.992123,0.610833


In [20]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Simple frequency encoding for categorical variables."""
    def __init__(self, cols=None):
        self.cols = cols

    def fit(self, X, y=None):
        X = X.copy()
        self.cols_ = self.cols if self.cols is not None else X.select_dtypes(exclude=np.number).columns.tolist()
        self.maps_ = {col: X[col].value_counts(normalize=True, dropna=False) for col in self.cols_}
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols_:
            X[col] = X[col].map(self.maps_[col]).fillna(0.0)
        return X

In [21]:
# Compare encodings using the same predictive model (Ridge)

ordinal_preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), cat_cols)
], verbose_feature_names_out=False)

all_cols = feature_cols.copy()
frequency_preprocessor = Pipeline([
    ('freq', FrequencyEncoder(cols=cat_cols)),
    ('preprocess', ColumnTransformer([
        ('all_numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), all_cols)
    ], verbose_feature_names_out=False))
])

encoding_models = {
    'Ridge + OneHot': Pipeline([
        ('preprocess', ohe_preprocessor),
        ('model', Ridge(alpha=ridge_search.best_params_['model__alpha']))
    ]),
    'Ridge + Ordinal': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', Ridge(alpha=ridge_search.best_params_['model__alpha']))
    ]),
    'Ridge + Frequency': Pipeline([
        ('preprocess', frequency_preprocessor),
        ('model', Ridge(alpha=ridge_search.best_params_['model__alpha']))
    ]),
}

rows = []
for name, est in encoding_models.items():
    summary = cv_summary(est, X, y, cv=cv, scoring=scoring).mean()
    summary['model'] = name
    rows.append(summary)


encoding_compare = pd.DataFrame(rows).set_index('model')#.sort_values('test_rmse')
encoding_compare


,rmse,mae,mape,r2
model,,,,
Ridge + OneHot,2.041264,0.976611,0.948883,0.679105
Ridge + Ordinal,2.260268,1.236579,1.238380,0.605053
Ridge + Frequency,2.328857,1.207039,1.442469,0.583382


## 8. Анализ коэффициентов и sparsity

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Sparsity: how many non-zero coefficients remain after regularization?
lasso_final = Pipeline([
    ('preprocess', ohe_preprocessor),
    ('model', Lasso(alpha=0.01, max_iter=20000))
])
lasso_final.fit(X_train, y_train)

feature_names = lasso_final.named_steps['preprocess'].get_feature_names_out()
coefficients = lasso_final.named_steps['model'].coef_
coef_table = pd.DataFrame({'feature': feature_names, 'coef': coefficients})
coef_table['abs_coef'] = coef_table['coef'].abs()
print('Non-zero coefficients:', (coef_table['coef'] != 0).sum(), 'out of', len(coef_table))
coef_table.sort_values('abs_coef', ascending=False).head(20)


Non-zero coefficients: 28 out of 6679


,feature,coef,abs_coef
49,model_G-Class,7.867122,7.867122
1,power_hp,1.767628,1.767628
19,brand_group_Europe,0.836657,0.836657
2,mileage_km,-0.833195,0.833195
90,fuel_type_бензин,-0.801682,0.801682
97,transmission_механика,0.776160,0.776160
4,age,-0.702393,0.702393
50,model_GLE-Class,0.619691,0.619691
18,brand_group_China,-0.602866,0.602866
6468,attributes_raw_4.0 л (585 л.с.) | бензин | роб...,0.533724,0.533724


## 9. Error analysis

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Error analysis on the hold-out test set
best_model_name = encoding_compare.index[0]
final_model = encoding_models[best_model_name]
final_model.fit(X_train, y_train)

test_pred = final_model.predict(X_test)
error_df = X_test.copy()
error_df['y_true'] = y_test.values
error_df['y_pred'] = test_pred
error_df['abs_error'] = np.abs(error_df['y_true'] - error_df['y_pred'])
error_df['pct_error'] = error_df['abs_error'] / np.maximum(np.abs(error_df['y_true']), 1)

error_df.sort_values('abs_error', ascending=False).head(10)


,brand,brand_group,brand_country,model,engine_volume_l,power_hp,fuel_type,transmission,drive,mileage_km,body_type,city,is_damaged,attributes_raw,age,y_true,y_pred,abs_error,pct_error
2705,Mercedes-Benz,Europe,Germany,G-Class,4.0,585.0,бензин,робот,4WD,15,SUV или внедорожник,Москва,0,4.0 л (585 л.с.) | бензин | робот | 4WD | 15 км,0,33.990000,16.621714,17.368286,0.510982
2741,Mercedes-Benz,Europe,Germany,G-Class,4.0,585.0,бензин,робот,4WD,97,SUV или внедорожник,Москва,0,4.0 л (585 л.с.) | бензин | робот | 4WD | 97 км,0,33.290000,16.473984,16.816016,0.505137
2695,Mercedes-Benz,Europe,Germany,G-Class,4.0,585.0,бензин,робот,4WD,21,SUV или внедорожник,Москва,0,4.0 л (585 л.с.) | бензин | робот | 4WD | 21 км,0,33.300000,16.529712,16.770288,0.503612
2713,Mercedes-Benz,Europe,Germany,G-Class,4.0,585.0,бензин,робот,4WD,17,SUV или внедорожник,Москва,0,4.0 л (585 л.с.) | бензин | робот | 4WD | 17 км,0,31.700000,16.514305,15.185695,0.479044
2793,Mercedes-Benz,Europe,Germany,G-Class,4.0,585.0,бензин,робот,4WD,20,SUV или внедорожник,Москва,0,4.0 л (585 л.с.) | бензин | робот | 4WD | 20 км,0,31.900000,16.762680,15.137320,0.474524
2783,Mercedes-Benz,Europe,Germany,G-Class,4.0,585.0,бензин,робот,4WD,15,SUV или внедорожник,Москва,0,4.0 л (585 л.с.) | бензин | робот | 4WD | 15 км,1,30.470000,16.542481,13.927519,0.457090
2610,Mercedes-Benz,Europe,Germany,G-Class,4.0,585.0,бензин,робот,4WD,5676,SUV или внедорожник,Москва,0,4.0 л (585 л.с.) | бензин | робот | 4WD | 5 67...,1,28.950000,16.348666,12.601334,0.435279
2702,Mercedes-Benz,Europe,Germany,G-Class,3.0,367.0,дизель,АКПП,4WD,37,SUV или внедорожник,Москва,0,3.0 л (367 л.с.) | дизель | АКПП | 4WD | 37 км,1,23.750000,12.299198,11.450802,0.482139
2720,Mercedes-Benz,Europe,Germany,G-Class,4.0,422.0,бензин,АКПП,4WD,2996,SUV или внедорожник,Москва,0,4.0 л (422 л.с.) | бензин | АКПП | 4WD | 2 996 км,10,22.677999,11.971233,10.706766,0.472121
2666,Mercedes-Benz,Europe,Germany,G-Class,4.0,585.0,бензин,АКПП,4WD,21365,SUV или внедорожник,Москва,0,4.0 л (585 л.с.) | бензин | АКПП | 4WD | 21 36...,3,26.050000,15.752598,10.297402,0.395294
